# 07 · Robustness controls for E6 (distractors) and E7 (paraphrases)

**Runtime:** any GPU (A100 or L4 is fastest, a T4 also works). **Run after:** 05 (needs the three saved main models). **Time:** about 30 to 45 min on an A100, longer on a T4. Evaluation only, nothing is retrained.

| Section | What it does |
|---|---|
| A | Matched paraphrase test: base and E7 models on exactly the same claims, original vs paraphrase (CPU, from saved predictions) |
| B | Builds three distractor conditions (random, same entity, topically related) with 3 samples each, filtered to MNLI-neutral sentences |
| C | Checks the distractors: added length, truncation at 128 tokens, word overlap with the claim, all split by label |
| D | Scores the 3 saved main models on the clean set and the 9 distractor sets |
| E | Summary tables, saved to `MyDrive/neural-evidence/controls/` |

Every step caches its output in Drive, so after a disconnect just run all again.

In [ ]:
# Setup: Drive, code from GitHub, packages, paths
from google.colab import drive
drive.mount('/content/drive')
import os, sys, json, shutil
BASE = '/content/drive/MyDrive/neural-evidence'
REPO_URL = 'https://github.com/sesha456/csce5218.git'
if not os.path.exists('/content/code'):
    !git clone -q {REPO_URL} /content/code
else:
    !git -C /content/code pull -q
CODE = '/content/code/neural-evidence'
os.chdir(CODE); sys.path.insert(0, CODE)
!pip -q install -r requirements.txt sentencepiece protobuf
PROC, RESULTS, RAW = f'{BASE}/data/processed', f'{BASE}/results', f'{BASE}/data/raw/fever'
CTRL, OUT = f'{PROC}/robustness/controls', f'{BASE}/controls'
for d in (CTRL, OUT): os.makedirs(d, exist_ok=True)
import numpy as np, pandas as pd
from ne import runtime, controls, metrics
from ne.data import read_jsonl, write_jsonl
runtime.setup()
SEEDS, N_SAMPLES = (42, 43, 44), 3
LAB = np.array(['S', 'R', 'NEI'])
test = read_jsonl(f'{PROC}/fever/test.jsonl')
y = np.array([r['label'] for r in test])
print(len(test), 'rows in the FEVER labelled dev evaluation set')

## A · Matched paraphrase test
The earlier E6/E7 numbers compared the full FEVER set with the 11K paraphrased subset. Here both models are scored on the same claims: each paraphrased claim against its own original.

In [ ]:
para = read_jsonl(f'{PROC}/robustness/fever_test_para.jsonl')
recs = []
for exp, run in [('MAIN', 'deberta_full'), ('E7', 'deberta_full_para_aug')]:
    for s in SEEDS:
        d = f'{RESULTS}/{exp}/{run}_seed{s}'
        m = controls.matched_paraphrase(test, para, np.load(f'{d}/preds_fever_test.npy'),
                                        np.load(f'{d}/preds_fever_test_para.npy'))
        m.pop('index')
        lo, hi = m.pop('delta_ci95')
        recs.append({'model': exp, 'seed': s, **m, 'delta_ci_lo': lo, 'delta_ci_hi': hi})
para_df = pd.DataFrame(recs).round(4)
para_df.to_csv(f'{OUT}/matched_paraphrase_by_seed.csv', index=False)
display(para_df)
cols = ['orig_macro_f1', 'para_macro_f1', 'delta', 'flip_rate']
para_sum = para_df.groupby('model', sort=False)[cols].agg(['mean', 'std']).round(4)
display(para_sum)
w = para_df.pivot(index='seed', columns='model', values=['orig_macro_f1', 'para_macro_f1'])
gain = pd.DataFrame({'E7 - MAIN on originals': w['orig_macro_f1']['E7'] - w['orig_macro_f1']['MAIN'],
                     'E7 - MAIN on paraphrases': w['para_macro_f1']['E7'] - w['para_macro_f1']['MAIN']})
display(gain.describe().loc[['mean', 'std']].round(4))

## B · Distractor conditions
One sentence is appended to the evidence of every claim, whatever its label, in three ways: **random** (another claim's single gold sentence from an unrelated page), **entity** (a random sentence from the same page as the claim's own evidence) and **topical** (the closest BM25 matches on the claim's evidence and retrieved pages). Sentences from any annotated evidence set, the claim's own evidence and its retrieved sentences are excluded, and only sentences that `roberta-large-mnli` labels neutral towards the claim are kept. Sample *j* uses the *j*-th surviving candidate.

In [ ]:
cand_path = f'{CTRL}/candidates_neutral.json'
if not os.path.exists(cand_path):
    from ne.wiki import WikiDB
    from ne.zeroshot import NLIScorer
    import torch
    local_db = '/content/wiki.db'
    if not os.path.exists(local_db):
        print('copying wiki.db to local disk (a few minutes)'); shutil.copy(f'{BASE}/data/wiki/wiki.db', local_db)
    ann = controls.annotated_sentences(read_jsonl(f'{RAW}/shared_task_dev.jsonl'))
    raw = controls.candidates(test, WikiDB(local_db), ann, n_cand=6, seed=0)
    scorer = NLIScorer('roberta-large-mnli')
    kept, share = controls.neutral_filter(raw, [r['claim'] for r in test], scorer)
    n_raw = {c: float(np.mean([len(x[c]) for x in raw])) for c in controls.CONDITIONS}
    json.dump({'kept': kept, 'neutral_share': share, 'raw_per_claim': n_raw}, open(cand_path, 'w'))
    del scorer; torch.cuda.empty_cache()
C = json.load(open(cand_path)); kept = C['kept']
print('candidates per claim before the MNLI filter:', C['raw_per_claim'])
print('share judged neutral:', C['neutral_share'])
common = controls.coverage_mask(kept, N_SAMPLES)
cov = pd.DataFrame({c: [np.mean([len(kept[i][c]) >= N_SAMPLES for i in np.where(y == k)[0]]) for k in range(3)]
                    for c in controls.CONDITIONS}, index=LAB)
cov['common'] = [common[y == k].mean() for k in range(3)]
print(f'claims with {N_SAMPLES} samples in every condition: {common.sum()} of {len(test)}')
display(cov.round(3))
files = {}
for c in controls.CONDITIONS:
    for j in range(N_SAMPLES):
        p = f'{CTRL}/fever_test_{c}_{j}.jsonl'
        if not os.path.exists(p):
            write_jsonl(controls.distracted_rows(test, kept, c, j), p)
        files[f'{c}_{j}'] = p
for c in controls.CONDITIONS:
    r = next(r for r in read_jsonl(files[f'{c}_0']) if r['has_distractor'] and r['label'] == 0)
    print(f'\n[{c}] {r["claim"]}\n   added: {r["distractor"][:200]}')

## C · Distractor checks: length, truncation at 128 tokens, overlap with the claim (by label)

In [ ]:
from transformers import AutoTokenizer
tok = AutoTokenizer.from_pretrained(f'{RESULTS}/MAIN/deberta_full_seed42/model')
L0 = controls.token_lengths(tok, test)
LEN, HAS, diag = {}, {}, []
for name, p in files.items():
    rows = read_jsonl(p)
    L = controls.token_lengths(tok, rows); has = np.array([r['has_distractor'] for r in rows])
    LEN[name], HAS[name] = L, has
    own_pages = [{q for q, _ in controls.own_ids(r)} for r in rows]
    for k in (*range(3), None):
        m = common & ((y == k) if k is not None else True)
        sel = np.where(m)[0]
        diag.append({'set': name, 'label': 'all' if k is None else LAB[k], 'n': int(m.sum()),
                     'extra_tokens': float((L - L0)[m].mean()),
                     'clean_over_128': float((L0[m] > 128).mean()),
                     'distractor_truncated': float((L[m] > 128).mean()),
                     'distractor_fully_cut': float((L0[m] >= 128).mean()),
                     'claim_word_overlap': float(np.mean([controls.claim_overlap(rows[i]['claim'], rows[i]['distractor']) for i in sel])),
                     'same_page_as_evidence': float(np.mean([rows[i]['distractor_ids'][0] in own_pages[i] for i in sel]))})
diag = pd.DataFrame(diag)
diag['condition'] = diag.set.str.rsplit('_', n=1).str[0]
diag_sum = diag.groupby(['condition', 'label'], sort=False).mean(numeric_only=True).round(3).reset_index()
diag.to_csv(f'{OUT}/distractor_checks.csv', index=False)
display(diag_sum)

## D · Score the three saved main models (clean + 9 distractor sets)

In [ ]:
import torch
from transformers import AutoModelForSequenceClassification
from ne.train import RunConfig, evaluate_model
EVAL = {'clean': f'{PROC}/fever/test.jsonl', **files}
for s in SEEDS:
    cfg = RunConfig(exp='CONTROLS', name='deberta_full', seed=s, model_name='', train_file='', results_dir=RESULTS)
    todo = {k: v for k, v in EVAL.items() if not os.path.exists(f'{cfg.run_dir}/preds_{k}.npy')}
    if not todo:
        print('skip seed', s); continue
    mdir = f'{RESULTS}/MAIN/deberta_full_seed{s}/model'
    tok_s = AutoTokenizer.from_pretrained(mdir)
    model = AutoModelForSequenceClassification.from_pretrained(mdir).float().to('cuda' if torch.cuda.is_available() else 'cpu')
    os.makedirs(cfg.run_dir, exist_ok=True)
    evaluate_model(model, tok_s, cfg, todo, cfg.run_dir)
    del model; torch.cuda.empty_cache()
for s in SEEDS:  # sanity check: the clean predictions should match notebook 05's
    a = np.load(f'{RESULTS}/CONTROLS/deberta_full_seed{s}/preds_clean.npy').argmax(-1)
    b = np.load(f'{RESULTS}/MAIN/deberta_full_seed{s}/preds_fever_test.npy').argmax(-1)
    print(f'seed {s}: clean predictions agree with 05 on {np.mean(a == b):.4f} of rows')

## E · Summary

In [ ]:
recs = []
for s in SEEDS:
    d = f'{RESULTS}/CONTROLS/deberta_full_seed{s}'
    pc = np.load(f'{d}/preds_clean.npy').argmax(-1)
    for name in ['clean', *files]:
        pr = np.load(f'{d}/preds_{name}.npy').argmax(-1)
        cond, j = ('clean', 0) if name == 'clean' else (name.rsplit('_', 1)[0], int(name.rsplit('_', 1)[1]))
        subsets = {'common': common}
        if name != 'clean':
            subsets['all with distractor'] = HAS[name]
            subsets['common, fits in 128'] = common & (LEN[name] <= 128)
        for sub, m in subsets.items():
            recs.append({'seed': s, 'condition': cond, 'sample': j, 'subset': sub,
                         'clean_f1_same_rows': metrics.evaluate(y[m], pc[m])['macro_f1'],
                         **controls.condition_stats(y, pc, pr, m)})
R = pd.DataFrame(recs)
R.to_csv(f'{OUT}/controls_all.csv', index=False)
order = ['clean', *controls.CONDITIONS]
def summ(sub):
    g = R[R.subset == sub].groupby('condition')
    t = g.agg(n=('n', 'first'), clean_f1=('clean_f1_same_rows', 'mean'), macro_f1=('macro_f1', 'mean'),
              f1_std=('macro_f1', 'std'), S_recall=('S_recall', 'mean'), R_recall=('R_recall', 'mean'),
              NEI_recall=('NEI_recall', 'mean'), pred_NEI=('pred_nei_share', 'mean'),
              flip_rate=('flip_rate', 'mean'), SR_to_NEI=('SR_to_NEI', 'mean'))
    sample_std = R[R.subset == sub].groupby(['condition', 'seed']).macro_f1.std().groupby('condition').mean()
    t['std_across_samples'] = sample_std
    return t.reindex([c for c in order if c in t.index]).round(4).reset_index()
tabs = {sub: summ(sub) for sub in ('common', 'common, fits in 128', 'all with distractor')}
for sub, t in tabs.items():
    print(f'\n== {sub} (mean over 3 seeds x 3 distractor samples) =='); display(t)
old = [json.load(open(f'{RESULTS}/MAIN/deberta_full_seed{s}/metrics_fever_test_distract_sent.json'))['macro_f1'] for s in SEEDS
       if os.path.exists(f'{RESULTS}/MAIN/deberta_full_seed{s}/metrics_fever_test_distract_sent.json')]
md_out = ['# Robustness controls (notebook 07)',
          f'Earlier one-sentence distractor set (notebook 05): macro-F1 {np.mean(old):.3f} (n seeds = {len(old)})' if old else '',
          '## Distractors, rows covered by every condition and sample\n\n' + tabs['common'].to_markdown(index=False),
          '## Distractors, same rows, only inputs that fit in 128 tokens\n\n' + tabs['common, fits in 128'].to_markdown(index=False),
          '## Distractors, every row that has a distractor in that condition\n\n' + tabs['all with distractor'].to_markdown(index=False),
          '## Distractor checks (common rows)\n\n' + diag_sum.to_markdown(index=False),
          '## Coverage by label\n\n' + cov.round(3).to_markdown(),
          '## Matched paraphrase test (same claims, original vs paraphrase)\n\n' + para_sum.to_markdown(),
          '## E7 minus MAIN, same claims\n\n' + gain.describe().loc[['mean', 'std']].round(4).to_markdown(),
          '## Matched paraphrase by seed\n\n' + para_df.to_markdown(index=False)]
open(f'{OUT}/controls_summary.md', 'w').write('\n\n'.join(x for x in md_out if x))
print('saved controls_summary.md, controls_all.csv, distractor_checks.csv, matched_paraphrase_by_seed.csv to', OUT)